# PyMODI+ 실습 커리큘럼
## 7차시 ~ 12차시: LED, 센서, 모터 제어

## 셀 1: 라이브러리 설치 (최초 1회 실행)
코랩 환경에 파이모디 및 키보드 제어 라이브러리를 설치합니다.

In [ ]:
# 라이브러리 설치
!pip install pymodi-plus
!pip install pynput

## 7차시: LED 및 디스플레이 기초
색상 순환 및 카운트다운 기능을 실습합니다.

In [ ]:
import modi_plus
import time

# BLE 연결 (UUID를 본인의 장치 번호로 수정하세요)
bundle = modi_plus.MODIPlus(connection_type="ble", network_uuid="본인의_UUID_입력")
led = bundle.leds
display = bundle.displays

# 실습: 5초 카운트다운 후 색상 순환
for i in range(5, 0, -1):
    display.text = f"  {i}"
    led.rgb = (255, 50*i, 0)
    time.sleep(1)

display.text = "GO!"
colors = [(255,0,0), (0,255,0), (0,0,255)] # 빨, 초, 파
for col in colors:
    led.rgb = col
    time.sleep(1)

led.turn_off()
display.reset()

## 8차시: 버튼 클릭 감지 및 환경 센서 모니터링
버튼 입력을 감지하고 주변 환경 값(빛, 소리, 온도, 습도)을 출력합니다.

In [ ]:
import modi_plus
import time

bundle = modi_plus.MODIPlus(connection_type="ble", network_uuid="본인의_UUID_입력")
button = bundle.buttons
env = bundle.envs
led = bundle.leds

print("버튼을 한 번 클릭하면 주변 환경 값을 출력합니다. 더블 클릭하면 종료합니다.")
while True:
    if button.double_clicked:
        print("프로그램 종료")
        break
    if button.clicked:
        print(f"빛: {env.illuminance:.1f}% | 소리: {env.volume:.1f}%")
        print(f"온도: {env.temperature:.1f}C | 습도: {env.humidity:.1f}%")
        led.rgb = (0, 255, 128)
        time.sleep(0.5)
        led.turn_off()
    time.sleep(0.1)

## 9차시: ToF 거리 경보 및 다이얼 밝기 제어
ToF 센서로 거리를 측정하고, 다이얼 입력으로 LED 밝기를 제어합니다.

In [ ]:
import modi_plus
import time

bundle = modi_plus.MODIPlus(connection_type="ble", network_uuid="본인의_UUID_입력")
tof = bundle.tofs
dial = bundle.dials
led = bundle.leds
display = bundle.displays

print("거리 경보 시스템 가동 중... (버튼 모듈이 있다면 클릭 시 종료)")
while True:
    dist = tof.distance
    val = dial.turn
    brightness = int(val * 2.55)
    
    # 다이얼로 LED 기본 밝기 조절
    led.rgb = (brightness, brightness, brightness)
    
    # ToF 거리 센서 조건
    if dist < 10:
        display.text = "DANGER!"
        led.rgb = (255, 0, 0)
    elif dist < 30:
        display.text = f"Dist: {dist:.0f}cm"
    else:
        display.text = "SAFE"
    
    # 버튼이 연결되어 있다면 종료 로직 (선택사항)
    if bundle.buttons and bundle.buttons.clicked: break
    time.sleep(0.1)

## 10차시: 5단계 스마트 경보 시스템
환경 센서 값을 조합하여 우선순위 기반 경보 시스템을 구현합니다.

**우선순위:**
1. 긴급: 어둡고 시끄러움
2. 고온 경고
3. 어두움 알림
4. 소음 알림
5. 정상

In [ ]:
import modi_plus
import time

bundle = modi_plus.MODIPlus(connection_type="ble", network_uuid="본인의_UUID_입력")
env = bundle.envs
led = bundle.leds
display = bundle.displays

def blink(color, n=3):
    for _ in range(n):
        led.rgb = color
        time.sleep(0.2); led.turn_off(); time.sleep(0.2)

print("5단계 스마트 경보 시스템 시작")
for _ in range(50): # 50회 루프
    li, so, te = env.illuminance, env.volume, env.temperature
    
    if li < 30 and so > 50: # 1순위: 긴급 (어둡고 시끄러움)
        blink((255, 0, 0))
        display.text = "EMERGENCY!"
    elif te > 35:           # 2순위: 고온 경고
        led.rgb = (255, 0, 0)
        display.text = f"HOT! {te:.0f}C"
    elif li < 30:           # 3순위: 어두움 알림
        led.rgb = (255, 255, 0)
        display.text = "DARK"
    elif so > 50:           # 4순위: 소음 알림
        blink((0, 0, 255))
        display.text = "LOUD"
    else:                   # 기본 상태
        led.rgb = (0, 255, 0)
        display.text = "ALL OK"
    time.sleep(0.5)

## 11차시: 무선 리모컨 제어
터미널 입력을 통해 LED와 디스플레이를 원격 제어합니다.

**명령:**
- r: 빨강
- g: 초록
- b: 파랑
- off: 끄기
- q: 종료

In [ ]:
import modi_plus

bundle = modi_plus.MODIPlus(connection_type="ble", network_uuid="본인의_UUID_입력")
led = bundle.leds
display = bundle.displays

print("명령을 입력하세요: r(빨강), g(초록), b(파랑), off(끄기), q(종료)")
while True:
    cmd = input("입력: ").lower()
    if cmd == "r":
        led.rgb, display.text = (255, 0, 0), "RED"
    elif cmd == "g":
        led.rgb, display.text = (0, 255, 0), "GREEN"
    elif cmd == "b":
        led.rgb, display.text = (0, 0, 255), "BLUE"
    elif cmd == "off":
        led.turn_off(); display.reset()
    elif cmd == "q":
        print("무선 제어 종료"); break

## 12차시-A: 모터 제어 (사각형 주행 패턴)
2개 이상의 모터로 사각형 경로를 주행합니다.

In [ ]:
import modi_plus
import time

bundle = modi_plus.MODIPlus(connection_type="ble", network_uuid="본인의_UUID_입력")

if len(bundle.motors) >= 2:
    m_l = bundle.motors[0]
    m_r = bundle.motors[1]

    def drive(ls, rs, sec):
        m_l.speed, m_r.speed = ls, rs
        time.sleep(sec)

    print("사각형 주행을 시작합니다.")
    for i in range(4):
        print(f"{i+1}번째 변 주행 중...")
        drive(50, 50, 2)    # 2초 직진
        drive(50, -50, 0.8)  # 0.8초 우회전
    
    drive(0, 0, 0) # 정지
else:
    print("모터 모듈이 2개 이상 연결되어야 합니다.")

## 12차시-B: 키보드 자동차 제어 (WASD)
**주의:** 코랩은 웹 브라우저 기반이므로 로컬 키보드 입력을 가로채기 위해 pynput 사용 시 **로컬 런타임 연결**이 필요할 수 있습니다.

**조종:**
- W: 전진
- S: 후진
- A: 좌회전
- D: 우회전
- Q: 종료

In [ ]:
import modi_plus
from pynput import keyboard

bundle = modi_plus.MODIPlus(connection_type="ble", network_uuid="본인의_UUID_입력")
m_l = bundle.motors[0]
m_r = bundle.motors[1]
SPD = 60

def on_press(key):
    try:
        k = key.char
        if k == "w": m_l.speed, m_r.speed = SPD, SPD   # 전진
        elif k == "s": m_l.speed, m_r.speed = -SPD, -SPD # 후진
        elif k == "a": m_l.speed, m_r.speed = -SPD, SPD  # 좌회전
        elif k == "d": m_l.speed, m_r.speed = SPD, -SPD  # 우회전
        elif k == "q": return False # 종료
    except: pass

def on_release(key):
    m_l.speed, m_r.speed = 0, 0 # 키 떼면 정지

print("WASD로 조종하세요. Q를 누르면 종료됩니다.")
with keyboard.Listener(on_press=on_press, on_release=on_release) as listener:
    listener.join()